In [1]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_theft_info_fixed(xml_file):
    """XML에서 theft_start, theft_end 프레임 번호 올바르게 추출"""
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        theft_start = None
        theft_end = None
        
        # track 요소들을 순회하면서 theft_start/end 찾기
        for track in root.findall('.//track'):
            label = track.get('label')
            
            if label == 'theft_start':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    theft_start = int(box.get('frame'))
            
            elif label == 'theft_end':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    theft_end = int(box.get('frame'))
        
        return theft_start, theft_end
    
    except Exception as e:
        print(f"❌ XML 파싱 오류 ({xml_file}): {e}")
        return None, None

def extract_theft_frames_fixed(video_dir, xml_dir, output_dir):
    """수정된 theft 구간 프레임 추출"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 비디오 파일 목록
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"총 {len(video_files)}개 비디오에서 theft 구간 추출 시작...")
    
    total_theft_frames = 0
    videos_with_theft = 0
    videos_without_theft = 0
    
    for video_file in tqdm(video_files, desc="비디오 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # 대응하는 XML 파일 찾기
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        if not os.path.exists(xml_path):
            print(f"⚠️ XML 파일 없음: {xml_file}")
            continue
        
        # theft 정보 추출 (수정된 함수)
        theft_start, theft_end = parse_theft_info_fixed(xml_path)
        
        if theft_start is None or theft_end is None:
            videos_without_theft += 1
            continue
        
        # 비디오 열기
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 {video_file}")
        print(f"   theft 구간: {theft_start} ~ {theft_end}")
        
        # theft 구간 유효성 검사
        if theft_end >= total_frames:
            print(f"   ⚠️ theft_end({theft_end})가 총 프레임({total_frames})보다 큼")
            theft_end = total_frames - 1
        
        frame_count = 0
        saved_frames = 0
        
        # 프레임별로 읽기
        while frame_count < total_frames:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # theft 구간에 있는 프레임만 저장
            if theft_start <= frame_count <= theft_end:
                # 파일명: 비디오이름_프레임번호_theft.jpg
                output_filename = f"{video_name}_{frame_count:03d}_theft.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                # JPG로 저장
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
            
            frame_count += 1
        
        cap.release()
        
        if saved_frames > 0:
            videos_with_theft += 1
            total_theft_frames += saved_frames
            print(f"   ✅ {saved_frames}개 theft 프레임 저장")
    
    print(f"\n🎉 추출 완료!")
    print(f"   theft 있는 비디오: {videos_with_theft}개")
    print(f"   theft 없는 비디오: {videos_without_theft}개") 
    print(f"   총 theft 프레임: {total_theft_frames}개")
    
    return total_theft_frames

def extract_normal_frames_fixed(video_dir, xml_dir, output_dir, sampling_rate=10):
    """
    정상 구간(theft가 아닌 구간) 프레임을 샘플링해서 저장 (수정된 버전)
    sampling_rate: N프레임마다 1개씩 저장 (기본값: 10프레임마다 1개)
    파일명: 비디오이름_프레임번호_normal.jpg
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal_frames = 0
    
    print(f"총 {len(video_files)}개 비디오에서 정상 구간 추출 시작... ({sampling_rate}프레임당 1개)")
    
    for video_file in tqdm(video_files, desc="정상 구간 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # XML에서 theft 구간 정보
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        # 수정된 파싱 함수 사용
        theft_start, theft_end = parse_theft_info_fixed(xml_path) if os.path.exists(xml_path) else (None, None)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0
        saved_normal = 0
        
        while frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 정상 구간 판단 (theft 구간이 아닌 곳)
            is_normal = True
            if theft_start is not None and theft_end is not None:
                if theft_start <= frame_count <= theft_end:
                    is_normal = False
            
            # 정상 구간이고 샘플링 조건에 맞으면 저장
            if is_normal and frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_normal += 1
            
            frame_count += 1
        
        cap.release()
        total_normal_frames += saved_normal
        
        if saved_normal > 0 and theft_start is not None:
            print(f"   ✅ {video_file}: {saved_normal}개 정상 프레임")
    
    print(f"\n🎉 정상 구간 추출 완료!")
    print(f"   총 정상 프레임: {total_normal_frames}개")
    
    return total_normal_frames

# =============================================================================
# 완전한 실행 코드 (theft + normal)
# =============================================================================

# 경로 설정
video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/train/video"
xml_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/train/label"
theft_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/train/theft_images"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/train/normal_images"

print("🚀 절도 감지용 데이터셋 구성 시작!")
print("="*50)

# 1단계: theft 구간 프레임 추출 (수정된 함수)
print("\n📍 1단계: theft 구간 추출")
theft_frames = extract_theft_frames_fixed(video_dir, xml_dir, theft_output_dir)

# 2단계: 정상 구간 프레임 추출 (수정된 함수)
print("\n📍 2단계: 정상 구간 추출")
normal_frames = extract_normal_frames_fixed(video_dir, xml_dir, normal_output_dir, sampling_rate=10)

# 최종 결과
print("\n" + "="*50)
print("🎉 데이터셋 구성 완료!")
print(f"📊 최종 결과:")
print(f"   🔴 Theft 프레임: {theft_frames:,}개")
print(f"   🟢 Normal 프레임: {normal_frames:,}개")
print(f"   📈 총 프레임: {theft_frames + normal_frames:,}개")
if theft_frames > 0:
    print(f"   ⚖️ 비율 (normal:theft): {normal_frames//theft_frames}:1")
print("="*50)

🚀 절도 감지용 데이터셋 구성 시작!

📍 1단계: theft 구간 추출
총 641개 비디오에서 theft 구간 추출 시작...


비디오 처리:   0%|          | 0/641 [00:00<?, ?it/s]


📹 C_3_12_22_BU_SMA_09-27_11-42-30_CC_RGB_DF2_M3.mp4
   theft 구간: 110 ~ 148


비디오 처리:   0%|          | 1/641 [00:03<39:14,  3.68s/it]

   ✅ 39개 theft 프레임 저장

📹 C_3_12_14_BU_SYB_09-28_14-28-05_CC_RGB_DF2_F2.mp4
   theft 구간: 76 ~ 140


...

   ✅ C_3_12_6_BU_SMA_08-30_14-48-20_CC_RGB_DF2_F1.mp4: 13개 정상 프레임


정상 구간 처리: 100%|█████████▉| 640/641 [18:29<00:01,  1.74s/it]

   ✅ C_3_12_7_BU_SMB_09-01_13-46-36_CB_RGB_DF2_M2.mp4: 8개 정상 프레임


정상 구간 처리: 100%|██████████| 641/641 [18:31<00:00,  1.73s/it]

   ✅ C_3_12_30_BU_SYB_10-04_14-43-54_CD_RGB_DF2_F3.mp4: 11개 정상 프레임

🎉 정상 구간 추출 완료!
   총 정상 프레임: 7319개

🎉 데이터셋 구성 완료!
📊 최종 결과:
   🔴 Theft 프레임: 43,357개
   🟢 Normal 프레임: 7,319개
   📈 총 프레임: 50,676개
   ⚖️ 비율 (normal:theft): 0:1


In [2]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_theft_info_fixed(xml_file):
    """XML에서 theft_start, theft_end 프레임 번호 올바르게 추출"""
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        theft_start = None
        theft_end = None
        
        # track 요소들을 순회하면서 theft_start/end 찾기
        for track in root.findall('.//track'):
            label = track.get('label')
            
            if label == 'theft_start':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    theft_start = int(box.get('frame'))
            
            elif label == 'theft_end':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    theft_end = int(box.get('frame'))
        
        return theft_start, theft_end
    
    except Exception as e:
        print(f"❌ XML 파싱 오류 ({xml_file}): {e}")
        return None, None

def extract_theft_frames_fixed(video_dir, xml_dir, output_dir):
    """수정된 theft 구간 프레임 추출"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 비디오 파일 목록
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"총 {len(video_files)}개 비디오에서 theft 구간 추출 시작...")
    
    total_theft_frames = 0
    videos_with_theft = 0
    videos_without_theft = 0
    
    for video_file in tqdm(video_files, desc="비디오 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # 대응하는 XML 파일 찾기
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        if not os.path.exists(xml_path):
            print(f"⚠️ XML 파일 없음: {xml_file}")
            continue
        
        # theft 정보 추출 (수정된 함수)
        theft_start, theft_end = parse_theft_info_fixed(xml_path)
        
        if theft_start is None or theft_end is None:
            videos_without_theft += 1
            continue
        
        # 비디오 열기
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 {video_file}")
        print(f"   theft 구간: {theft_start} ~ {theft_end}")
        
        # theft 구간 유효성 검사
        if theft_end >= total_frames:
            print(f"   ⚠️ theft_end({theft_end})가 총 프레임({total_frames})보다 큼")
            theft_end = total_frames - 1
        
        frame_count = 0
        saved_frames = 0
        
        # 프레임별로 읽기
        while frame_count < total_frames:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # theft 구간에 있는 프레임만 저장
            if theft_start <= frame_count <= theft_end:
                # 파일명: 비디오이름_프레임번호_theft.jpg
                output_filename = f"{video_name}_{frame_count:03d}_theft.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                # JPG로 저장
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
            
            frame_count += 1
        
        cap.release()
        
        if saved_frames > 0:
            videos_with_theft += 1
            total_theft_frames += saved_frames
            print(f"   ✅ {saved_frames}개 theft 프레임 저장")
    
    print(f"\n🎉 추출 완료!")
    print(f"   theft 있는 비디오: {videos_with_theft}개")
    print(f"   theft 없는 비디오: {videos_without_theft}개") 
    print(f"   총 theft 프레임: {total_theft_frames}개")
    
    return total_theft_frames

def extract_normal_frames_fixed(video_dir, xml_dir, output_dir, sampling_rate=10):
    """
    정상 구간(theft가 아닌 구간) 프레임을 샘플링해서 저장 (수정된 버전)
    sampling_rate: N프레임마다 1개씩 저장 (기본값: 10프레임마다 1개)
    파일명: 비디오이름_프레임번호_normal.jpg
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal_frames = 0
    
    print(f"총 {len(video_files)}개 비디오에서 정상 구간 추출 시작... ({sampling_rate}프레임당 1개)")
    
    for video_file in tqdm(video_files, desc="정상 구간 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # XML에서 theft 구간 정보
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        # 수정된 파싱 함수 사용
        theft_start, theft_end = parse_theft_info_fixed(xml_path) if os.path.exists(xml_path) else (None, None)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0
        saved_normal = 0
        
        while frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 정상 구간 판단 (theft 구간이 아닌 곳)
            is_normal = True
            if theft_start is not None and theft_end is not None:
                if theft_start <= frame_count <= theft_end:
                    is_normal = False
            
            # 정상 구간이고 샘플링 조건에 맞으면 저장
            if is_normal and frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_normal += 1
            
            frame_count += 1
        
        cap.release()
        total_normal_frames += saved_normal
        
        if saved_normal > 0 and theft_start is not None:
            print(f"   ✅ {video_file}: {saved_normal}개 정상 프레임")
    
    print(f"\n🎉 정상 구간 추출 완료!")
    print(f"   총 정상 프레임: {total_normal_frames}개")
    
    return total_normal_frames

# =============================================================================
# 완전한 실행 코드 (theft + normal)
# =============================================================================

# 경로 설정
video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/val/video"
xml_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/val/label"
theft_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/val/theft_images"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/stealing_behavior/val/normal_images"

print("🚀 절도 감지용 데이터셋 구성 시작!")
print("="*50)

# 1단계: theft 구간 프레임 추출 (수정된 함수)
print("\n📍 1단계: theft 구간 추출")
theft_frames = extract_theft_frames_fixed(video_dir, xml_dir, theft_output_dir)

# 2단계: 정상 구간 프레임 추출 (수정된 함수)
print("\n📍 2단계: 정상 구간 추출")
normal_frames = extract_normal_frames_fixed(video_dir, xml_dir, normal_output_dir, sampling_rate=10)

# 최종 결과
print("\n" + "="*50)
print("🎉 데이터셋 구성 완료!")
print(f"📊 최종 결과:")
print(f"   🔴 Theft 프레임: {theft_frames:,}개")
print(f"   🟢 Normal 프레임: {normal_frames:,}개")
print(f"   📈 총 프레임: {theft_frames + normal_frames:,}개")
if theft_frames > 0:
    print(f"   ⚖️ 비율 (normal:theft): {normal_frames//theft_frames}:1")
print("="*50)

🚀 절도 감지용 데이터셋 구성 시작!

📍 1단계: theft 구간 추출
총 80개 비디오에서 theft 구간 추출 시작...


비디오 처리:   0%|          | 0/80 [00:00<?, ?it/s]


📹 C_3_12_45_BU_DYB_10-16_15-02-02_CD_RGB_DF2_F1.mp4
   theft 구간: 85 ~ 128


비디오 처리:   1%|▏         | 1/80 [00:03<04:37,  3.52s/it]

   ✅ 44개 theft 프레임 저장

📹 C_3_12_47_BU_SMC_10-14_16-15-55_CA_RGB_DF2_M3.mp4
   theft 구간: 87 ~ 160


...

   ✅ C_3_12_45_BU_SMC_10-14_16-07-18_CB_RGB_DF2_M3.mp4: 11개 정상 프레임


정상 구간 처리:  99%|█████████▉| 79/80 [02:11<00:01,  1.73s/it]

   ✅ C_3_12_48_BU_DYB_10-17_13-46-19_CA_RGB_DF2_M3.mp4: 12개 정상 프레임


정상 구간 처리: 100%|██████████| 80/80 [02:13<00:00,  1.67s/it]

   ✅ C_3_12_40_BU_SMC_10-14_11-43-44_CE_RGB_DF2_M2.mp4: 11개 정상 프레임

🎉 정상 구간 추출 완료!
   총 정상 프레임: 960개

🎉 데이터셋 구성 완료!
📊 최종 결과:
   🔴 Theft 프레임: 4,832개
   🟢 Normal 프레임: 960개
   📈 총 프레임: 5,792개
   ⚖️ 비율 (normal:theft): 0:1


In [4]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def extract_purchase_normal_frames(video_dir, output_dir, sampling_rate=10):
    """
    purchase_behavior 비디오에서 정상 프레임 추출 (수정된 버전)
    """
    
    # 출력 디렉토리 생성 (이 부분이 빠져있었음!)
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal = 0
    
    print(f"📂 발견된 비디오 파일: {len(video_files)}개")
    print(f"📁 출력 디렉토리: {output_dir}")
    
    for video_file in tqdm(video_files, desc="Purchase 정상 프레임 추출"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
            
        frame_count = 0
        saved_frames = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 10프레임마다 1개씩 저장
            if frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
                else:
                    print(f"❌ 이미지 저장 실패: {output_filename}")
            
            frame_count += 1
        
        cap.release()
        total_normal += saved_frames
        
        if saved_frames == 0:
            print(f"⚠️  {video_file}에서 저장된 프레임이 0개")
    
    return total_normal

# 실행
purchase_video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/purchase_behavior/train/video"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/purchase_behavior/train/normal_images"

additional_normal = extract_purchase_normal_frames(purchase_video_dir, normal_output_dir)
print(f"🛒 Purchase에서 추가된 Normal: {additional_normal:,}개")

📂 발견된 비디오 파일: 804개
📁 출력 디렉토리: /home/ckim/dev_ws/project_ws/mldl_project/data/purchase_behavior/train/normal_images


Purchase 정상 프레임 추출: 100%|██████████| 804/804 [1:22:13<00:00,  6.14s/it]

🛒 Purchase에서 추가된 Normal: 48,391개


In [5]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def extract_purchase_normal_frames(video_dir, output_dir, sampling_rate=10):
    """
    purchase_behavior 비디오에서 정상 프레임 추출 (수정된 버전)
    """
    
    # 출력 디렉토리 생성 (이 부분이 빠져있었음!)
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal = 0
    
    print(f"📂 발견된 비디오 파일: {len(video_files)}개")
    print(f"📁 출력 디렉토리: {output_dir}")
    
    for video_file in tqdm(video_files, desc="Purchase 정상 프레임 추출"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
            
        frame_count = 0
        saved_frames = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 10프레임마다 1개씩 저장
            if frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
                else:
                    print(f"❌ 이미지 저장 실패: {output_filename}")
            
            frame_count += 1
        
        cap.release()
        total_normal += saved_frames
        
        if saved_frames == 0:
            print(f"⚠️  {video_file}에서 저장된 프레임이 0개")
    
    return total_normal

# 실행
purchase_video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/purchase_behavior/val/video"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/purchase_behavior/val/normal_images"

additional_normal = extract_purchase_normal_frames(purchase_video_dir, normal_output_dir)
print(f"🛒 Purchase에서 추가된 Normal: {additional_normal:,}개")

📂 발견된 비디오 파일: 100개
📁 출력 디렉토리: /home/ckim/dev_ws/project_ws/mldl_project/data/purchase_behavior/val/normal_images


Purchase 정상 프레임 추출: 100%|██████████| 100/100 [11:30<00:00,  6.90s/it]

🛒 Purchase에서 추가된 Normal: 6,016개


: 